# Unidad 3 - Algoritmos de Clasificación 

**Dataset:** Breast Cancer Wisconsin (sklearn)

**Objetivos de la clase:**
1. Demostrar la importancia del escalado en Logistic Regression y SVM.
2. Comparar tres algoritmos: Logistic Regression, Random Forest, SVM.
3. Visualizar matriz de confusión, curva ROC, AUC, e importancia de features.
4. Mostrar el impacto del desbalanceo de clases y cómo corregirlo.
5. Guiar en la resolución del TP (aprobación de tarjetas de crédito).

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, roc_auc_score)

# Configuración de gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

print("[OK] Librerías cargadas.")

## 2. Carga y exploración del dataset

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')  # 0 = maligno, 1 = benigno

print(f"Dimensiones: {X.shape}")
print("\nDistribución del target:")
print(y.value_counts())
print(f"\nPorcentaje de benignos: {y.mean():.2%}")

### 2.1 Visualización rápida de algunas features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
features_to_plot = ['mean radius', 'mean texture', 'mean perimeter']

for i, feat in enumerate(features_to_plot):
    sns.histplot(data=X, x=feat, hue=y, kde=True, ax=axes[i], palette='Set1')
    axes[i].set_title(f'{feat} vs Target')
plt.tight_layout()
plt.show()

### 2.2 División en entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Entrenamiento: {X_train.shape[0]} muestras")
print(f"Prueba: {X_test.shape[0]} muestras")

## 3. Experimentos: Sin escalado vs Con escalado

**Observación:** Logistic Regression y SVM usan distancias / gradientes, por lo que **requieren** escalado. Random Forest no.

In [ ]:
# Diccionario para guardar resultados
results = {'modelo': [], 'accuracy': [], 'precision': [], 'recall': [], 'f1': []}

def register_metrics(name, y_true, y_pred):
    results['modelo'].append(name)
    results['accuracy'].append(accuracy_score(y_true, y_pred))
    results['precision'].append(precision_score(y_true, y_pred))
    results['recall'].append(recall_score(y_true, y_pred))
    results['f1'].append(f1_score(y_true, y_pred))

### Experimento 1: Logistic Regression SIN escalar

In [ ]:
print("\n" + "="*80)
print("EXPERIMENTO 1: Logistic Regression SIN escalar")
print("="*80)

lr_unscaled = LogisticRegression(max_iter=1000, random_state=42)
lr_unscaled.fit(X_train, y_train)
y_pred_lr_un = lr_unscaled.predict(X_test)

acc_lr_un = accuracy_score(y_test, y_pred_lr_un)
register_metrics('LR sin escalar', y_test, y_pred_lr_un)
print(f"Accuracy: {acc_lr_un:.4f}")

### Experimento 2: Logistic Regression CON escalado

In [ ]:
print("\n" + "="*80)
print("EXPERIMENTO 2: Logistic Regression CON escalado")
print("="*80)

pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
pipeline_lr.fit(X_train, y_train)
y_pred_lr_sc = pipeline_lr.predict(X_test)

acc_lr_sc = accuracy_score(y_test, y_pred_lr_sc)
register_metrics('LR con escalar', y_test, y_pred_lr_sc)
print(f"Accuracy: {acc_lr_sc:.4f}")
print(f"Mejora: {acc_lr_sc - acc_lr_un:.4f}")

### Experimento 3: Random Forest (no necesita escalado)

In [ ]:
print("\n" + "="*80)
print("EXPERIMENTO 3: Random Forest (100 árboles)")
print("="*80)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
register_metrics('Random Forest', y_test, y_pred_rf)
print(f"Accuracy: {acc_rf:.4f}")

### Experimento 4: SVM sin escalar (puede ser muy lento o no converger)

In [ ]:
print("\n" + "="*80)
print("EXPERIMENTO 4: SVM SIN escalar")
print("="*80)

svm_unscaled = SVC(kernel='rbf', random_state=42)
svm_unscaled.fit(X_train, y_train)
y_pred_svm_un = svm_unscaled.predict(X_test)

acc_svm_un = accuracy_score(y_test, y_pred_svm_un)
register_metrics('SVM sin escalar', y_test, y_pred_svm_un)
print(f"Accuracy: {acc_svm_un:.4f}")

### Experimento 5: SVM con escalado

In [ ]:
print("\n" + "="*80)
print("EXPERIMENTO 5: SVM CON escalado")
print("="*80)

pipeline_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(kernel='rbf', random_state=42))
])
pipeline_svm.fit(X_train, y_train)
y_pred_svm_sc = pipeline_svm.predict(X_test)

acc_svm_sc = accuracy_score(y_test, y_pred_svm_sc)
register_metrics('SVM con escalar', y_test, y_pred_svm_sc)
print(f"Accuracy: {acc_svm_sc:.4f}")
print(f"Mejora: {acc_svm_sc - acc_svm_un:.4f}")

## 4. Comparación visual de métricas

In [ ]:
df_results = pd.DataFrame(results)
df_results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
df_results_melted = df_results.melt(id_vars='modelo', var_name='métrica', value_name='valor')
sns.barplot(data=df_results_melted, x='modelo', y='valor', hue='métrica', ax=ax)
ax.set_title('Comparación de métricas por modelo', fontsize=14)
ax.set_ylim(0, 1)
ax.legend(loc='lower right')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print("[CONCLUSIÓN] El escalado mejora drásticamente LR y SVM. Random Forest ya funciona bien sin escalar.")

## 5. Matriz de confusión para el mejor modelo (LR con escalado)

In [ ]:
print("\n" + "="*80)
print("MATRIZ DE CONFUSIÓN - Logistic Regression con escalado")
print("="*80)

cm = confusion_matrix(y_test, y_pred_lr_sc)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Maligno', 'Benigno'],
            yticklabels=['Maligno', 'Benigno'])
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de Confusión')
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr_sc, target_names=['Maligno', 'Benigno']))

## 6. Curva ROC y AUC

In [ ]:
print("\n" + "="*80)
print("CURVA ROC - Comparación de modelos")
print("="*80)

# Para LR y RF necesitamos las probabilidades
y_prob_lr = pipeline_lr.predict_proba(X_test)[:, 1]
y_prob_rf = rf.predict_proba(X_test)[:, 1]
y_prob_svm = pipeline_svm.decision_function(X_test)  # SVM no da probabilidades directas, usamos decision_function

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_prob_svm)

auc_lr = roc_auc_score(y_test, y_prob_lr)
auc_rf = roc_auc_score(y_test, y_prob_rf)
auc_svm = roc_auc_score(y_test, y_prob_svm)

plt.figure(figsize=(10, 8))
plt.plot(fpr_lr, tpr_lr, label=f'LR (AUC = {auc_lr:.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'RF (AUC = {auc_rf:.3f})')
plt.plot(fpr_svm, tpr_svm, label=f'SVM (AUC = {auc_svm:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Azar (AUC = 0.5)')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curva ROC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("[CONCLUSIÓN] Los tres modelos tienen AUC > 0.98, lo que indica excelente capacidad predictiva.")

## 7. Importancia de features (Random Forest) y coeficientes (LR)

In [ ]:
print("\n" + "="*80)
print("IMPORTANCIA DE FEATURES")
print("="*80)

# Importancia en Random Forest
importances = rf.feature_importances_
feature_names = X.columns
indices = np.argsort(importances)[::-1][:10]  # Top 10

plt.figure(figsize=(10, 6))
plt.barh([feature_names[i] for i in indices][::-1], importances[indices][::-1], color='skyblue')
plt.xlabel('Importancia')
plt.title('Top 10 features más importantes (Random Forest)')
plt.tight_layout()
plt.show()

print("\nCoeficientes de Logistic Regression (escalado):")
coefs = pipeline_lr.named_steps['classifier'].coef_[0]
coef_df = pd.DataFrame({'feature': feature_names, 'coef': coefs})
coef_df = coef_df.reindex(coef_df.coef.abs().sort_values(ascending=False).index).head(10)
print(coef_df)

## 8. Simulación de desbalanceo y corrección con class_weight

In [ ]:
print("\n" + "="*80)
print("EJEMPLO DE DESBALANCEO Y CORRECCIÓN")
print("="*80)

# Creamos un dataset artificialmente desbalanceado (solo 10% de malignos)
np.random.seed(42)
indices_malignant = np.where(y_train == 0)[0]
indices_benign = np.where(y_train == 1)[0]
# Reducimos los malignos a 20% del total
n_malignant = int(0.2 * len(indices_benign))
indices_malignant_sampled = np.random.choice(indices_malignant, n_malignant, replace=False)
indices_train_imbalanced = np.concatenate([indices_malignant_sampled, indices_benign])
np.random.shuffle(indices_train_imbalanced)

X_train_imb = X_train.iloc[indices_train_imbalanced]
y_train_imb = y_train.iloc[indices_train_imbalanced]

print(f"Nuevo tamaño de entrenamiento: {len(X_train_imb)}")
print(f"Proporción de malignos: {y_train_imb.mean():.2%}")

# Entrenamos LR sin class_weight
lr_imb = LogisticRegression(max_iter=1000, random_state=42)
lr_imb.fit(X_train_imb, y_train_imb)
y_pred_imb = lr_imb.predict(X_test)

# Entrenamos LR con class_weight='balanced'
lr_imb_balanced = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_imb_balanced.fit(X_train_imb, y_train_imb)
y_pred_imb_bal = lr_imb_balanced.predict(X_test)

print("\nMétricas SIN class_weight:")
print(classification_report(y_test, y_pred_imb, target_names=['Maligno', 'Benigno']))

print("\nMétricas CON class_weight='balanced':")
print(classification_report(y_test, y_pred_imb_bal, target_names=['Maligno', 'Benigno']))

print("[CONCLUSIÓN] El class_weight mejora el Recall de la clase minoritaria (Maligno) a costa de algo de precisión.")

## 9. Uso de IA (ChatGPT) para interpretar resultados

**Prompt sugerido:**
*"Tengo un modelo de clasificación binaria (cáncer de mama) con los siguientes resultados: [copiar classification report]. ¿Qué métrica debo priorizar si quiero minimizar los falsos negativos? ¿Qué técnica me recomiendas para mejorar el Recall?"*

**Discusión en clase:** La IA puede sugerir ajustar el umbral de decisión, usar SMOTE, o cambiar el algoritmo. Esto refuerza que la IA es un copiloto, no un sustituto del criterio humano.

## 10. Resumen final (Takeaways)

1. **Escalado:** Imprescindible para Logistic Regression y SVM. No necesario para Random Forest.
2. **Métricas:** Elegir según el problema. Si los FN son costosos → priorizar Recall. Si los FP son costosos → priorizar Precision. F1-Score para balancear ambos.
3. **Curva ROC / AUC:** Herramienta visual para evaluar la capacidad de discriminación del modelo. AUC > 0.8 es bueno.
4. **Desbalanceo:** Usar `class_weight='balanced'` o técnicas de remuestreo (SMOTE) para mejorar el rendimiento en clases minoritarias.
5. **Importancia de features:** Ayuda a entender qué variables impulsan las predicciones (y a veces a simplificar el modelo).

## 11. Guía para el Trabajo Práctico (Aprobación de Tarjetas de Crédito)

**Dataset:** (se proporcionará en la plataforma)

**Pasos recomendados:**
1. **Exploración:** `df.head()`, `df.info()`, `df.describe()`, `df['target'].value_counts()`.
2. **Limpieza:** Imputar nulos (con media, mediana o moda según la variable).
3. **Feature Engineering:** One-Hot Encoding para variables categóricas (como se hizo en Titanic).
4. **Escalado:** Aplicar `StandardScaler` a las variables numéricas si usas LR o SVM.
5. **Modelos:** Probar al menos Logistic Regression y Random Forest. Opcional: SVM.
6. **Evaluación:** Usar classification report, matriz de confusión y curva ROC.
7. **Conclusión:** Elegir el mejor modelo y justificar la elección con métricas.

**Recomendación:** Si el dataset está desbalanceado, probar `class_weight='balanced'`.